# Modeling, Segmentation & Evaluation

## Overview
This notebook focuses on building, comparing, and evaluating machine learning models for the Bank Marketing dataset. Building on the preprocessing and feature engineering steps from previous notebooks, we now develop predictive models to estimate the likelihood of a client subscribing to a term deposit.

In addition to a traditional global modeling approach, this notebook explores a segmentation-based strategy using clusters derived earlier. By comparing a single global model against cluster-specific models, we aim to understand whether segment-level specialization leads to improved predictive performance and better business outcomes.



## Objectives
- Compare multiple candidate model architectures using the training and validation sets  
- Select a primary modeling approach based on validation performance  
- Train a global model using all available data and evaluate performance across clusters  
- Train segment-specific models for each cluster and compare against the global model  
- Perform hyperparameter tuning for both global and segment-specific models using validation data  
- Train final models using selected hyperparameters  
- Apply probability calibration to improve predicted probabilities  
- Conduct cost-sensitive analysis using calibrated probabilities and different decision thresholds  
- Evaluate final models on a held-out test set, including overall and cluster-level performance  



## Dataset Description
The modeling process uses the preprocessed dataset generated in the previous notebook, which includes engineered features such as categorical encodings, binary indicators, and interaction terms. Additionally, each observation is assigned to a cluster derived from unsupervised learning, enabling both global and segment-specific modeling approaches.

The data is split into three subsets:

- **Training set**: used to fit models  
- **Validation set**: used for model selection, hyperparameter tuning, and calibration  
- **Test set**: held out for final unbiased evaluation  

The target variable represents whether a client subscribed to a term deposit.



## Key Considerations
- **Model comparison**: Different model architectures are evaluated to identify the most suitable approach for the problem  
- **Segmentation strategy**: Cluster-based segmentation allows us to investigate heterogeneity in client behavior and model performance  
- **Fairness and consistency**: Performance is evaluated across clusters to detect potential disparities  
- **Overfitting prevention**: Validation data is strictly used for model selection and hyperparameter tuning  
- **Probability calibration**: Raw model probabilities may be poorly calibrated, which can negatively impact decision-making  
- **Business alignment**: Model evaluation incorporates cost-sensitive analysis, considering trade-offs between false positives (e.g., unnecessary calls) and false negatives (missed opportunities)  
- **Final evaluation integrity**: The test set remains untouched until the final evaluation step  



## Outcome
By the end of this notebook, we will have:
- Selected and trained a primary global model  
- Developed segment-specific models tailored to different client clusters  
- Tuned and calibrated all models for improved predictive reliability  
- Compared global and segmented approaches from both predictive and business perspectives  
- Identified optimal decision thresholds based on expected cost  
- Produced a final evaluation on the test set, including overall performance, cluster-level insights, and cost-based metrics  

In [11]:
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.model_selection import ParameterGrid
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score

from utils.utils import load_dataset, evaluate_model, evaluate_from_probs

RANDOM_STATE = 705

In [ ]:
# quarto preview 03_training.ipynb --to pdf
# quarto render 03_training.ipynb
# black 03_training.ipynb

## 1. Global architecture selection

In this section, we evaluate a set of candidate model architectures to determine which will be used in the subsequent comparison between a global model (trained on all observations) and segment-specific models (trained per cluster).

We begin by training and assessing five baseline architectures using their default configurations:

1. Logistic Regression  
2. Random Forest  
3. XGBoost  
4. LightGBM  
5. CatBoost  

The goal is to identify a strong primary model that balances predictive performance and generalization before proceeding to more advanced analysis.

In [2]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# from sklearn.metrics import (
#     average_precision_score,
#     f1_score,
#     precision_score,
#     recall_score,
# )

from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

Because the outcome variable is highly imbalanced, we apply class weighting across all models to penalize misclassification of subscribers more heavily than non‑subscribers. For tree‑based models, this is implemented via the ratio of negative to positive samples, while linear and ensemble baselines use built‑in balanced class weights. This approach preserves the original data distribution and produces probability estimates suitable for downstream calibration and cost‑based decision analysis.

In [3]:
X_train_, y_train = load_dataset("02", "train")
X_validation_, y_validation = load_dataset("02", "validation")


train
X shape: (28934, 60)
y shape: (28934,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0



validation
X shape: (7234, 60)
y shape: (7234,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0


In [4]:
cluster_feat = "cluster"

X_train = X_train_.copy()
X_train = X_train.drop(columns=cluster_feat)
print("\nTrain set: ", X_train.shape)
display(X_train.head(3))

X_validation = X_validation_.copy()
X_validation = X_validation.drop(columns=cluster_feat).copy()
print("\nValidation set: ", X_validation.shape)
display(X_validation.head(3))


Train set:  (28934, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0



Validation set:  (7234, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [5]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

### i. Logistic Regression

In [6]:
log_reg = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

log_reg.fit(X_train, y_train)
log_reg_metrics = evaluate_model(log_reg, X_validation, y_validation)

### ii. Random Forest

In [7]:
rf = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)

rf.fit(X_train, y_train)
rf_metrics = evaluate_model(rf, X_validation, y_validation)

### iii. XGBoost

In [8]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
)

xgb.fit(X_train, y_train)
xgb_metrics = evaluate_model(xgb, X_validation, y_validation)

### iv. LightGBM

In [9]:
lgbm = lgb.LGBMClassifier(
    objective="binary", scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE
)

lgbm.fit(X_train, y_train)
lgbm_metrics = evaluate_model(lgbm, X_validation, y_validation)

[LightGBM] [Info] Number of positive: 3385, number of negative: 25549
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001658 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 176
[LightGBM] [Info] Number of data points in the train set: 28934, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116990 -> initscore=-2.021244
[LightGBM] [Info] Start training from score -2.021244


### v. CatBoost

In [10]:
cat = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    class_weights=[1, scale_pos_weight],
    verbose=0,
    random_state=RANDOM_STATE,
)

cat.fit(X_train, y_train)
cat_metrics = evaluate_model(cat, X_validation, y_validation)

### Results comparison

In [13]:
results = pd.DataFrame.from_dict(
    {
        "Logistic Regression": log_reg_metrics,
        "Random Forest": rf_metrics,
        "XGBoost": xgb_metrics,
        "LightGBM": lgbm_metrics,
        "CatBoost": cat_metrics,
    },
    orient="index",
)

results.sort_values(by="AUC_PR", ascending=False)

,AUC_PR,F1,Precision,Recall
LightGBM,0.445832,0.459201,0.362826,0.625296
CatBoost,0.442484,0.459831,0.368159,0.612293
Logistic Regression,0.411748,0.395126,0.287325,0.632388
XGBoost,0.407729,0.434281,0.347795,0.578014
Random Forest,0.313136,0.275808,0.413712,0.206856


On the validation set, LightGBM and CatBoost exhibit very similar performance, with nearly identical F1 scores and only a marginal difference in AUC‑PR (0.446 for LightGBM versus 0.442 for CatBoost). Both models clearly outperform the other architectures and appear well suited to handling the strong class imbalance present in the data.

LightGBM was selected as the primary model architecture because it achieves the highest AUC‑PR overall, indicating slightly better ranking of likely subscribers, while maintaining strong recall and competitive precision. Given the minimal performance gap, this choice is pragmatic rather than definitive; CatBoost remains a strong alternative and could be expected to perform similarly in downstream analysis. **LightGBM** is therefore used for the remainder of the project to provide a consistent baseline for both global and segment‑specific modeling.

## 2. Global model

Once we decided to work with LightGBM, we'll train the global model which will the observations from all clusters.

In [2]:
X_train_, y_train = load_dataset("02", "train")
X_validation_, y_validation = load_dataset("02", "validation")
# X_test_, y_test_ = load_dataset(nb="02", split="test")


train
X shape: (28934, 60)
y shape: (28934,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0



validation
X shape: (7234, 60)
y shape: (7234,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0


In [3]:
cluster_feat = "cluster"

X_train = X_train_.copy()
X_train = X_train.drop(columns=cluster_feat)
print("\nTrain set: ", X_train.shape)
display(X_train.head(3))

X_validation = X_validation_.copy()
X_validation = X_validation.drop(columns=cluster_feat).copy()
print("\nValidation set: ", X_validation.shape)
display(X_validation.head(3))

# X_test = X_test_.copy()
# X_test = X_test.drop(columns=cluster_feat).copy()
# print("\nTest set: ", X_test.shape)
# display(X_test.head(3))


Train set:  (28934, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0



Validation set:  (7234, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [4]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

print(f"Scale pos weight: {scale_pos_weight:.2f}")

Scale pos weight: 7.55


For hyperparameter tuning, we'll use a Bayesian approach

In [5]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "aucpr",
        "random_state": RANDOM_STATE,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        # hyperparameters to tune
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_val_prob = model.predict_proba(X_validation)[:, 1]
    aucpr = average_precision_score(y_validation, y_val_prob)

    return aucpr

In [6]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC-PR:", study.best_value)
print("Best parameters:", study.best_params)

best_params = study.best_params

[I 2026-04-10 22:48:24,296] A new study created in memory with name: no-name-e2787aa4-49a5-4f4b-9498-28ca916dc756
Best trial: 0. Best value: 0.446455:   3%|▎         | 1/30 [00:00<00:25,  1.13it/s]

[I 2026-04-10 22:48:25,210] Trial 0 finished with value: 0.4464554093510928 and parameters: {'num_leaves': 26, 'max_depth': 7, 'learning_rate': 0.1125253685438908, 'n_estimators': 202, 'min_child_samples': 127}. Best is trial 0 with value: 0.4464554093510928.


Best trial: 1. Best value: 0.454725:   7%|▋         | 2/30 [00:01<00:22,  1.23it/s]

[I 2026-04-10 22:48:25,968] Trial 1 finished with value: 0.4547250845261611 and parameters: {'num_leaves': 79, 'max_depth': 4, 'learning_rate': 0.06243348344891831, 'n_estimators': 324, 'min_child_samples': 22}. Best is trial 1 with value: 0.4547250845261611.


Best trial: 1. Best value: 0.454725:  10%|█         | 3/30 [00:02<00:24,  1.11it/s]

...

[I 2026-04-10 22:49:11,121] Trial 26 finished with value: 0.450809625055938 and parameters: {'num_leaves': 87, 'max_depth': 3, 'learning_rate': 0.1312720479582124, 'n_estimators': 381, 'min_child_samples': 139}. Best is trial 1 with value: 0.4547250845261611.
[I 2026-04-10 22:49:11,294] Trial 27 finished with value: 0.41280974026825507 and parameters: {'num_leaves': 36, 'max_depth': 1, 'learning_rate': 0.170704146226309, 'n_estimators': 316, 'min_child_samples': 75}. Best is trial 1 with value: 0.4547250845261611.


Best trial: 1. Best value: 0.454725:  97%|█████████▋| 29/30 [00:47<00:00,  1.48it/s]

[I 2026-04-10 22:49:12,234] Trial 28 finished with value: 0.45187442810261563 and parameters: {'num_leaves': 102, 'max_depth': 5, 'learning_rate': 0.03706309018792351, 'n_estimators': 343, 'min_child_samples': 199}. Best is trial 1 with value: 0.4547250845261611.


Best trial: 1. Best value: 0.454725: 100%|██████████| 30/30 [00:49<00:00,  1.63s/it]

[I 2026-04-10 22:49:13,356] Trial 29 finished with value: 0.4344204853299442 and parameters: {'num_leaves': 84, 'max_depth': 7, 'learning_rate': 0.11829643596585648, 'n_estimators': 207, 'min_child_samples': 126}. Best is trial 1 with value: 0.4547250845261611.
Best AUC-PR: 0.4547250845261611
Best parameters: {'num_leaves': 79, 'max_depth': 4, 'learning_rate': 0.06243348344891831, 'n_estimators': 324, 'min_child_samples': 22}


We train the final global base model with the best parameters

In [7]:
global_model = lgb.LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    **best_params
)

global_model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,79
,max_depth,4
,learning_rate,0.06243348344891831
,n_estimators,324
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,22


Now we do the probabilities calibration

In [8]:
# raw probabilities on validation
val_probs_raw = global_model.predict_proba(X_validation)[:, 1]

# isotonic regression calibration
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs_raw, y_validation)

# calibrated probabilities
val_probs_calibrated = calibrator.transform(val_probs_raw)

In [9]:
global_validation_metrics = evaluate_from_probs(y_validation, val_probs_calibrated)

print("Global model (validation, calibrated):")
print(global_validation_metrics)

Global model (validation, calibrated):
{'AUC_PR': 0.44398317702417234, 'F1': 0.4176157934700076, 'Precision': 0.583864118895966, 'Recall': 0.32505910165484636}


In [12]:
cluster_results = {}

for c in sorted(X_validation_[cluster_feat].unique()):
    idx = X_validation_[cluster_feat] == c

    cluster_results[f"cluster_{c}"] = evaluate_from_probs(
        y_validation.loc[idx], val_probs_calibrated[idx]
    )

cluster_results_df = pd.DataFrame(cluster_results).T
cluster_results_df

,AUC_PR,F1,Precision,Recall
cluster_0,0.567553,0.512195,0.617647,0.437500
cluster_1,0.285366,0.294118,0.522388,0.204678
cluster_2,0.367921,0.359155,0.510000,0.277174
cluster_3,0.505459,0.449679,0.625000,0.351171


## 3. Cluster 0 model

In [ ]:
cluster_id = 0
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)

## 4. Cluster 1 model

In [ ]:
cluster_id = 1
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)

## 5. Cluster 2 model

In [ ]:
cluster_id = 2
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)

## 6. Cluster 3 model

In [ ]:
cluster_id = 3
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)